# Hotel Booking — Feature Engineering---## OverviewThis notebook transforms the raw hotel booking dataset into a modeling-ready feature set. Building on insights from the EDA phase, we apply systematic feature creation, transformation, encoding, and selection to maximize predictive power for cancellation forecasting.**Input**: Cleaned hotel_booking.csv (119,390 records × 36 columns)  **Output**: Engineered feature matrix ready for model training  **Target Variable**: `is_canceled` (Binary: 0/1)---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("hotel_booking.csv")

# ============================================================
# STEP 1: INITIAL CLEANING (from EDA recommendations)
# ============================================================
print("STEP 1: APPLYING EDA-RECOMMENDED CLEANING\n")

# Handle missing values
df['children'] = df['children'].fillna(0).astype(int)
df['country'] = df['country'].fillna('Unknown')
df['agent'] = df['agent'].fillna(0).astype(int)

# Drop unusable columns
drop_cols = ['company', 'name', 'email', 'phone-number', 'credit_card',
             'reservation_status', 'reservation_status_date']
df.drop(columns=drop_cols, inplace=True)
print(f"  Dropped {len(drop_cols)} columns: {drop_cols}")

# Remove invalid records
initial = len(df)
df = df[(df['adults'] + df['children'] + df['babies']) > 0]  # zero-guest
df = df[df['adr'] >= 0]  # negative ADR
df = df[df['adr'] < 5000]  # extreme ADR
print(f"  Removed {initial - len(df):,} invalid records")
print(f"  Clean dataset: {len(df):,} rows × {df.shape[1]} columns")


STEP 1: APPLYING EDA-RECOMMENDED CLEANING

  Dropped 7 columns: ['company', 'name', 'email', 'phone-number', 'credit_card', 'reservation_status', 'reservation_status_date']
  Removed 182 invalid records
  Clean dataset: 119,208 rows × 29 columns


---## Step 2: Derived Feature CreationCreating new features based on domain knowledge and EDA patterns to capture booking complexity, guest profiles, revenue potential, and temporal signals.

In [ ]:
# ============================================================
# STEP 2: FEATURE ENGINEERING — NEW DERIVED FEATURES
# ============================================================
print("STEP 2: CREATING DERIVED FEATURES\n")

# --- Stay Duration Features ---
df['total_stay'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['weekend_ratio'] = np.where(df['total_stay'] > 0,
                               df['stays_in_weekend_nights'] / df['total_stay'], 0)
df['is_weekend_stay'] = (df['stays_in_weekend_nights'] > 0).astype(int)

# --- Guest Composition Features ---
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df['is_family'] = ((df['children'] > 0) | (df['babies'] > 0)).astype(int)
df['is_solo'] = (df['total_guests'] == 1).astype(int)
df['is_couple'] = ((df['adults'] == 2) & (df['children'] == 0) & (df['babies'] == 0)).astype(int)

# --- Revenue Features ---
df['total_revenue'] = df['adr'] * df['total_stay']
df['revenue_per_guest'] = np.where(df['total_guests'] > 0,
                                    df['total_revenue'] / df['total_guests'], 0)

# --- Lead Time Binning ---
df['lead_time_bin'] = pd.cut(df['lead_time'],
                              bins=[0, 7, 30, 90, 180, 365, 800],
                              labels=['last_minute', 'short', 'medium', 'long', 'very_long', 'extreme'],
                              include_lowest=True)

# --- Booking Complexity ---
df['booking_complexity'] = df['total_of_special_requests'] + df['booking_changes']
df['has_special_requests'] = (df['total_of_special_requests'] > 0).astype(int)
df['has_booking_changes'] = (df['booking_changes'] > 0).astype(int)

# --- Room Assignment Features ---
df['room_type_changed'] = (df['reserved_room_type'] != df['assigned_room_type']).astype(int)

# --- Guest History Features ---
df['has_previous_cancellations'] = (df['previous_cancellations'] > 0).astype(int)
df['has_previous_bookings'] = (df['previous_bookings_not_canceled'] > 0).astype(int)
df['cancellation_ratio'] = np.where(
    (df['previous_cancellations'] + df['previous_bookings_not_canceled']) > 0,
    df['previous_cancellations'] / (df['previous_cancellations'] + df['previous_bookings_not_canceled']),
    0)

# --- Agent Feature ---
df['has_agent'] = (df['agent'] > 0).astype(int)
df['is_direct_booking'] = ((df['distribution_channel'] == 'Direct') | (df['agent'] == 0)).astype(int)

# --- Temporal Features ---
month_map = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
             'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
df['arrival_month_num'] = df['arrival_date_month'].map(month_map)
df['is_summer'] = df['arrival_month_num'].isin([6,7,8]).astype(int)
df['is_peak_season'] = df['arrival_month_num'].isin([6,7,8,12]).astype(int)
df['quarter'] = pd.cut(df['arrival_month_num'], bins=[0,3,6,9,12], labels=['Q1','Q2','Q3','Q4'])

# --- Waiting List Feature ---
df['was_on_waitlist'] = (df['days_in_waiting_list'] > 0).astype(int)

# --- Parking Feature ---
df['needs_parking'] = (df['required_car_parking_spaces'] > 0).astype(int)

new_features = ['total_stay', 'weekend_ratio', 'is_weekend_stay', 'total_guests', 
                'is_family', 'is_solo', 'is_couple', 'total_revenue', 'revenue_per_guest',
                'lead_time_bin', 'booking_complexity', 'has_special_requests',
                'has_booking_changes', 'room_type_changed', 'has_previous_cancellations',
                'has_previous_bookings', 'cancellation_ratio', 'has_agent',
                'is_direct_booking', 'arrival_month_num', 'is_summer', 'is_peak_season',
                'quarter', 'was_on_waitlist', 'needs_parking']

print(f"  Created {len(new_features)} new features:")
for i, f in enumerate(new_features, 1):
    print(f"    {i:2d}. {f}")
print(f"\n  Dataset now: {df.shape[0]:,} rows × {df.shape[1]} columns")


STEP 2: CREATING DERIVED FEATURES

  Created 25 new features:
     1. total_stay
     2. weekend_ratio
     3. is_weekend_stay
     4. total_guests
     5. is_family
     6. is_solo
     7. is_couple
     8. total_revenue
     9. revenue_per_guest
    10. lead_time_bin
    11. booking_complexity
    12. has_special_requests
    13. has_booking_changes
    14. room_type_changed
    15. has_previous_cancellations
    16. has_previous_bookings
    17. cancellation_ratio
    18. has_agent
    19. is_direct_booking
    20. arrival_month_num
    21. is_summer
    22. is_peak_season
    23. quarter
    24. was_on_waitlist
    25. needs_parking

  Dataset now: 119,208 rows × 54 columns


---## Step 3: Categorical Encoding & Outlier Treatment

In [ ]:
# ============================================================
# STEP 3: ENCODING & OUTLIER HANDLING
# ============================================================
print("STEP 3: CATEGORICAL ENCODING & OUTLIER TREATMENT\n")

# --- Rare Category Grouping ---
for col in ['meal', 'market_segment', 'distribution_channel']:
    vc = df[col].value_counts(normalize=True)
    rare_cats = vc[vc < 0.01].index
    if len(rare_cats) > 0:
        df[col] = df[col].replace(rare_cats, 'Other')
        print(f"  {col}: grouped {list(rare_cats)} → 'Other'")

# --- Country Grouping (top 10 + Other) ---
top_countries = df['country'].value_counts().head(10).index
df['country_grouped'] = np.where(df['country'].isin(top_countries), df['country'], 'Other')
print(f"  country: kept top 10, grouped {df['country'].nunique()-10} others → 'Other'")

# --- Ordinal Encoding for Months ---
df['arrival_month_encoded'] = df['arrival_date_month'].map(month_map)

# --- One-Hot Encoding ---
encode_cols = ['hotel', 'meal', 'market_segment', 'distribution_channel',
               'deposit_type', 'customer_type', 'lead_time_bin', 'quarter', 'country_grouped']
df_encoded = pd.get_dummies(df, columns=encode_cols, drop_first=True, dtype=int)
print(f"\n  One-hot encoded {len(encode_cols)} columns")

# --- ADR Outlier Capping (IQR method) ---
Q1, Q3 = df_encoded['adr'].quantile(0.25), df_encoded['adr'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = max(0, Q1 - 1.5*IQR), Q3 + 1.5*IQR
original_outliers = ((df_encoded['adr'] < lower) | (df_encoded['adr'] > upper)).sum()
df_encoded['adr'] = df_encoded['adr'].clip(lower, upper)
print(f"  ADR: capped {original_outliers:,} outliers to [{lower:.0f}, {upper:.0f}]")

# --- Log Transform for Skewed Features ---
for col in ['lead_time', 'days_in_waiting_list', 'total_revenue']:
    df_encoded[f'{col}_log'] = np.log1p(df_encoded[col])
    print(f"  Created {col}_log (log transform)")

print(f"\n  Final encoded dataset: {df_encoded.shape[0]:,} rows × {df_encoded.shape[1]} columns")


STEP 3: CATEGORICAL ENCODING & OUTLIER TREATMENT

  meal: grouped ['Undefined', 'FB'] → 'Other'
  market_segment: grouped ['Complementary', 'Aviation', 'Undefined'] → 'Other'
  distribution_channel: grouped ['GDS', 'Undefined'] → 'Other'
  country: kept top 10, grouped 168 others → 'Other'

  One-hot encoded 9 columns
  ADR: capped 3,793 outliers to [0, 211]
  Created lead_time_log (log transform)
  Created days_in_waiting_list_log (log transform)
  Created total_revenue_log (log transform)

  Final encoded dataset: 119,208 rows × 78 columns


---## Step 4: Feature Selection & Correlation with Target

In [ ]:
# ============================================================
# STEP 4: FEATURE IMPORTANCE & SELECTION
# ============================================================
print("STEP 4: FEATURE-TARGET CORRELATION ANALYSIS\n")

# Get numeric columns only
num_cols = df_encoded.select_dtypes(include=[np.number]).columns
target_corr = df_encoded[num_cols].corr()['is_canceled'].drop('is_canceled').abs().sort_values(ascending=False)

print("Top 20 Features by |correlation| with is_canceled:\n")
for i, (col, r) in enumerate(target_corr.head(20).items(), 1):
    sign = df_encoded[num_cols].corr().loc['is_canceled', col]
    direction = "+" if sign > 0 else "-"
    bar = "█" * int(r * 50)
    print(f"  {i:2d}. {col:45s} |r|={r:.4f} ({direction}) {bar}")

# Drop features with near-zero correlation
weak_features = target_corr[target_corr < 0.01].index.tolist()
print(f"\n  Features with |r| < 0.01 (candidates for removal): {len(weak_features)}")

# Drop original columns that were encoded
cols_to_drop = ['arrival_date_month', 'reserved_room_type', 'assigned_room_type',
                'country', 'agent']
existing_drops = [c for c in cols_to_drop if c in df_encoded.columns]
df_encoded.drop(columns=existing_drops, inplace=True, errors='ignore')
print(f"  Dropped {len(existing_drops)} superseded columns")
print(f"\n  Final feature set: {df_encoded.shape[1]-1} features + 1 target")


STEP 4: FEATURE-TARGET CORRELATION ANALYSIS

Top 20 Features by |correlation| with is_canceled:

   1. deposit_type_Non Refund                   |r|=0.4685 (+) ███████████████████████
   2. lead_time                                 |r|=0.2931 (+) ██████████████
   3. total_of_special_requests                 |r|=0.2347 (-) ███████████
   4. has_special_requests                      |r|=0.2289 (-) ███████████
   5. required_car_parking_spaces               |r|=0.1955 (-) █████████
   6. needs_parking                             |r|=0.1948 (-) █████████
   7. lead_time_log                             |r|=0.1903 (+) █████████
   8. booking_changes                           |r|=0.1444 (-) ███████
   9. has_booking_changes                       |r|=0.1395 (-) ██████
  10. booking_complexity                        |r|=0.1217 (-) ██████
  11. previous_cancellations                    |r|=0.1101 (+) █████
  12. has_previous_cancellations                |r|=0.1078 (+) █████
  13. cancellation_r

---## Step 5: Final Feature Summary### Engineered Feature Categories| Category | Features | Purpose ||----------|----------|---------|| **Stay Duration** | total_stay, weekend_ratio, is_weekend_stay | Capture stay patterns || **Guest Profile** | total_guests, is_family, is_solo, is_couple | Guest type signals || **Revenue** | total_revenue, revenue_per_guest | Financial impact || **Booking Behavior** | lead_time_bin, booking_complexity, has_special_requests | Commitment level || **Room Assignment** | room_type_changed | Service quality proxy || **Guest History** | cancellation_ratio, has_previous_cancellations | Past behavior || **Channel** | has_agent, is_direct_booking | Booking source || **Temporal** | arrival_month_num, is_summer, is_peak_season, quarter | Seasonality || **Waiting List** | was_on_waitlist | Demand indicator |### Key Transformations Applied- **Missing values**: Imputed (children→0, country→Unknown, agent→0), company dropped- **Rare categories**: Grouped into 'Other' (threshold: <1%)- **Encoding**: One-hot for nominals, ordinal for months- **Outliers**: ADR capped using IQR method- **Log transforms**: lead_time, days_in_waiting_list, total_revenue- **Target leakage**: reservation_status removed- **PII**: All personal data columns dropped**The dataset is now ready for model training and evaluation.**

In [ ]:
# Save engineered dataset
print("FEATURE ENGINEERING COMPLETE\n")
print(f"  Original dataset:    119,390 rows × 36 columns")
print(f"  Engineered dataset:  {df_encoded.shape[0]:,} rows × {df_encoded.shape[1]} columns")
print(f"  New features added:  25")
print(f"  Columns removed:     12 (PII + leakage + unusable)")
print(f"  Target balance:      {(df_encoded['is_canceled'].mean()*100):.1f}% canceled")
print(f"\n✓ Dataset ready for model training!")


FEATURE ENGINEERING COMPLETE

  Original dataset:    119,390 rows × 36 columns
  Engineered dataset:  119,208 rows × 73 columns
  New features added:  25
  Columns removed:     12 (PII + leakage + unusable)
  Target balance:      37.0% canceled

✓ Dataset ready for model training!
